In [ ]:
 
import json
import os
import joblib
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import shutil

c:\Users\User\OneDrive\Desktop\Customer Churn Prediction Automation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ml_client = MLClient.from_config(credential=DefaultAzureCredential())

Found the config file in: C:\Users\User\OneDrive\Desktop\Customer Churn Prediction Automation\.azureml\config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [21]:
versions = ml_client.models.list(name="customer-churn-model")

In [ ]:
prod_version = next(v for v in versions if v.tags.get("stage") == "production")


In [63]:
MODEL_NAME   = "customer-churn-model"
DOWNLOAD_DIR = "../downloaded_production_model"

In [62]:
## Just to clear the existing folder before downloading the model again

if os.path.exists(DOWNLOAD_DIR):
    shutil.rmtree(DOWNLOAD_DIR)
    print(f"Cleared existing folder: {DOWNLOAD_DIR}")

ml_client.models.download(name=MODEL_NAME, version=prod_version.version, download_path="../downloaded_production_model")

Cleared existing folder: ./downloaded_production_model


In [64]:
model_dir    = os.path.join(DOWNLOAD_DIR, MODEL_NAME, "model")
model        = joblib.load(os.path.join(model_dir, "model.pkl"))
feature_cols = json.load(open(os.path.join(model_dir, "feature_columns.json")))

c:\Users\User\OneDrive\Desktop\Customer Churn Prediction Automation\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.5.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [26]:
df= pd.read_csv("../data/train.csv")

In [31]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.preprocessing import preprocess_data
X, y = preprocess_data(df)

In [36]:
X=X[feature_cols]

In [39]:
print(X.columns)

Index(['Age', 'Avg Monthly GB Download', 'Avg Monthly Long Distance Charges',
       'CLTV', 'Contract', 'Gender', 'Married', 'Monthly Charge',
       'Number of Dependents', 'Number of Referrals', 'Population',
       'Senior Citizen', 'Tenure in Months', 'Total Charges',
       'Total Extra Data Charges', 'Total Long Distance Charges',
       'Total Refunds', 'Total Revenue', 'Under 30', 'City Freq',
       'Internet Type_DSL', 'Internet Type_Fiber Optic',
       'Payment Method_Credit Card', 'Payment Method_Mailed Check',
       'Offer_Offer B', 'Offer_Offer C', 'Offer_Offer D', 'Offer_Offer E'],
      dtype='object')


In [ ]:
coef_df = pd.DataFrame({
    "feature":     feature_cols,
    "coefficient": model.coef_[0],
    "odds_ratio":  np.exp(model.coef_[0]),
}).sort_values("coefficient", key=abs, ascending=False).reset_index(drop=True)
 
print(f"\n{'Feature'} --------------------- {'Coefficient'}---------------------------{'Odds Ratio'}")
print("-" * 66)
for _, row in coef_df.head(15).iterrows():
    direction = " churn Increases" if row["coefficient"] > 0 else " churn Decreases"
    print(f"{row['feature']} ----------------- {row['coefficient']} ------------------ {row['odds_ratio']} which Results -  {direction}")

coef_df.to_csv('Coeffiecients.csv', index=False)


Feature --------------------- Coefficient---------------------------Odds Ratio
------------------------------------------------------------------
Number of Referrals ----------------- -0.17720465557311463 ------------------ 0.8376083457645955 which Results -   churn Decreases
Tenure in Months ----------------- -0.12367495066605172 ------------------ 0.8836670295843726 which Results -   churn Decreases
Number of Dependents ----------------- -0.05707946054337543 ------------------ 0.9445190143904963 which Results -   churn Decreases
Contract ----------------- -0.05685527395016675 ------------------ 0.9447307866279147 which Results -   churn Decreases
Payment Method_Credit Card ----------------- -0.022273488821947383 ------------------ 0.9779727338627041 which Results -   churn Decreases
Age ----------------- 0.019348730478532153 ------------------ 1.019537130301488 which Results -   churn Increases
Offer_Offer D ----------------- -0.014744403290728184 ------------------ 0.98536376315435

In [ ]:
import shap
 
explainer   = shap.LinearExplainer(model, shap.maskers.Independent(X))
shap_values = explainer.shap_values(X)
# shape: (n_customers, n_features)
 
shap_df = pd.DataFrame({
    "feature":       feature_cols,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
    "mean_shap":     shap_values.mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
 
print(f"feature  ------------------------------- Mean SHAP ------------------------------------------- Effect")
print("-" * 60)
for _, row in shap_df.head(15).iterrows():
    effect = "churn" if row["mean_shap"] > 0 else "stay"
    print(f"{row['feature']} ------------------- {round(row['mean_abs_shap'],3)}  --------------------      {effect}")

shap_df.to_csv('SHAP_Values.csv', index=False)

 

feature  ------------------------------- Mean SHAP ------------------------------------------- Effect
------------------------------------------------------------
Total Charges ------------------- 8.288  --------------------      stay
Total Revenue ------------------- 8.287  --------------------      churn
Tenure in Months ------------------- 2.811  --------------------      churn
Total Long Distance Charges ------------------- 2.169  --------------------      stay
Number of Referrals ------------------- 0.416  --------------------      stay
Age ------------------- 0.293  --------------------      stay
Avg Monthly GB Download ------------------- 0.12  --------------------      churn
Monthly Charge ------------------- 0.102  --------------------      stay
City Freq ------------------- 0.079  --------------------      stay
Total Refunds ------------------- 0.057  --------------------      stay
Avg Monthly Long Distance Charges ------------------- 0.044  --------------------      stay
Con

In [55]:

customer= int(input("\nEnter a customer index to explain"))

pred_proba = model.predict_proba(X.iloc[[customer]])[0][1]
 
local_df = pd.DataFrame({
    "feature":      feature_cols,
    "value":        X.iloc[0].values,
    "contribution": shap_values[0],
}).sort_values("contribution", key=abs, ascending=False)
 
print(f"\nPredicted churn probability: {pred_proba:.2%}")
print(f"\n{'Feature'} --------------------  {'Value'} ------------------------ {'Contribution'}")
print("-" * 65)
for _, row in local_df.head(10).iterrows():
    direction = "churn Increases" if row["contribution"] > 0 else "churn Decreases"
    print(f"{row['feature']} -------------- {row['value']} -------------------- {row['contribution']}  {direction}")


Predicted churn probability: 77.13%

Feature --------------------  Value ------------------------ Contribution
-----------------------------------------------------------------
Tenure in Months -------------- 58 -------------------- -2.8346298692659055  churn Decreases
Total Charges -------------- 3563.8 -------------------- 2.665012563500217  churn Increases
Total Revenue -------------- 4562.56 -------------------- -2.5066321421289  churn Decreases
Total Long Distance Charges -------------- 998.76 -------------------- 0.43893569703870333  churn Increases
Age -------------- 31 -------------------- -0.4047754416108927  churn Decreases
Number of Referrals -------------- 1 -------------------- 0.12581530545691139  churn Increases
City Freq -------------- 23 -------------------- 0.1158359743097554  churn Increases
Monthly Charge -------------- 60.3 -------------------- -0.1018962605547623  churn Decreases
Total Refunds -------------- 0.0 -------------------- -0.040775445052407766  churn D

In [66]:
for _, row in shap_df.head(5).iterrows():
    #safe = row["feature"].replace(" ", "_")
    print(row["feature"])

Total Charges
Total Revenue
Tenure in Months
Total Long Distance Charges
Number of Referrals
